In [ ]:
# GIL (Global Interpreter Lock)
# - CPython has a GIL that prevents true parallel thread execution
# - Only one thread executes Python bytecode at a time
# - Affects CPU-bound tasks (use multiprocessing instead)
# - I/O-bound tasks can still benefit from threading
# - Does NOT exist in Jython, IronPython
print("GIL: Only one thread runs Python code at a time in CPython")

In [ ]:
# Shallow vs Deep Copy
import copy

original = [[1, 2], [3, 4]]

# Shallow copy: outer list is new, inner lists are shared
shallow = copy.copy(original)
shallow[0].append(99)
print(original)  # [[1, 2, 99], [3, 4]] - AFFECTED!

# Deep copy: everything is independent
original = [[1, 2], [3, 4]]
deep = copy.deepcopy(original)
deep[0].append(99)
print(original)  # [[1, 2], [3, 4]] - NOT affected!

In [ ]:
# is vs ==
a = [1, 2, 3]
b = [1, 2, 3]
print(a == b)   # True (same value)
print(a is b)   # False (different objects)
# Use 'is' only for None, True, False singletons

In [ ]:
# Mutable default argument
def bad(lst=[]):
    lst.append(1)
    return lst

print(bad())   # [1]
print(bad())   # [1, 1]  - shared!

def good(lst=None):
    if lst is None:
        lst = []
    lst.append(1)
    return lst

print(good())  # [1]
print(good())  # [1]  - independent!

In [ ]:
# Integer caching: -5 to 256
a = 256; b = 256
print(a is b)  # True

# String interning
a = "hello"; b = "hello"
print(a is b)  # True (auto-interned)

In [ ]:
# Descriptors
class Validator:
    def __init__(self, min_val, max_val):
        self.min_val = min_val
        self.max_val = max_val

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        return getattr(obj, f'_{self.name}', None)

    def __set__(self, obj, value):
        if not self.min_val <= value <= self.max_val:
            raise ValueError(f"{self.name} must be between {self.min_val} and {self.max_val}")
        setattr(obj, f'_{self.name}', value)

class Student:
    grade = Validator(0, 100)
    def __init__(self, grade):
        self.grade = grade

s = Student(85)
print(s.grade)   # 85
# s.grade = 150  # ValueError!

In [ ]:
# Chained comparison
x = 5
print(1 < x < 10)     # True (same as: 1 < x and x < 10)
print(1 < 2 < 3 < 4)  # True

In [ ]:
# Walrus operator :=
data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
if (n := len(data)) > 5:
    print(f"Long list: {n} elements")

In [ ]:
# Ellipsis (...)
# Used as:
# 1. Placeholder (like pass)
def todo():
    ...

# 2. Type hints for variable-length tuples
from typing import Tuple
# Tuple[int, ...] = tuple of any number of ints

# 3. NumPy slicing (not core Python)
print(type(...))  # <class 'ellipsis'>

In [ ]:
# _ variable
# 1. Don't care variable
for _ in range(3):
    print("hi", end=" ")
print()

# 2. Last result in interactive interpreter
# >>> 2 + 3
# 5
# >>> _ * 2
# 10

# 3. Unpacking
a, _, c = (1, 2, 3)
print(a, c)  # 1 3

In [ ]:
# Multiple assignment trap
a = b = []   # SAME list!
a.append(1)
print(b)     # [1]

# Correct:
a = []
b = []

In [ ]:
# Truthy/Falsy complete list
# Falsy: 0, 0.0, 0j, "", [], (), {}, set(), None, False, range(0)
# Everything else is truthy

In [ ]:
# Python memory management
# 1. Reference counting (main mechanism)
# 2. Garbage collection (for circular references)
import sys
x = [1, 2, 3]
print(sys.getrefcount(x))  # Usually 2+ (one for x, one for getrefcount arg)

# weakref - reference that doesn't prevent garbage collection
import weakref
class MyObj:
    pass

obj = MyObj()
weak = weakref.ref(obj)
print(weak())  # <__main__.MyObj object>
del obj
print(weak())  # None (garbage collected)

In [ ]:
# functools.partial
from functools import partial

def power(base, exponent):
    return base ** exponent

square = partial(power, exponent=2)
cube = partial(power, exponent=3)

print(square(5))  # 25
print(cube(3))    # 27

In [ ]:
# operator module
import operator
print(operator.add(2, 3))     # 5
print(operator.mul(4, 5))     # 20

# itemgetter - great for sorting
from operator import itemgetter
data = [("Alice", 25), ("Bob", 20), ("Charlie", 30)]
print(sorted(data, key=itemgetter(1)))  # Sort by age

---
# 🔬 DEEP DIVE — Traps & Gotchas
---

In [ ]:
# TRAP: Augmented assignment (+=) behaves differently!
# For MUTABLE: modifies in-place
a = [1, 2]; b = a
b += [3]        # Same as b.extend([3])
print(a)        # [1, 2, 3] — affected!

# For IMMUTABLE: creates new object
a = (1, 2); b = a
b += (3,)       # Creates NEW tuple
print(a)        # (1, 2) — NOT affected!

# Also important with = ... +:
a = [1, 2]; b = a
b = b + [3]     # NEW list created
print(a)        # [1, 2] — NOT affected!

In [ ]:
# TRAP: Chained assignment
a = b = c = []  # ALL reference SAME list!
a.append(1)
print(b, c)  # [1] [1] — all affected!

# FIX:
a, b, c = [], [], []  # Different lists

In [ ]:
# TRAP: Dictionary ordering gotcha
d = {True: 'yes', 1: 'one', 1.0: 'float_one'}
print(d)  # {True: 'float_one'}
# Why? True == 1 == 1.0, all hash to same value
# Key stays as first inserted (True), value gets overwritten

---
# 🧪 INTERACTIVE EXAMPLES
---

In [ ]:
# What will this print?
x = [1, 2, 3]
y = x
x = x + [4]
print(y)  # ?

In [ ]:
# Answer: [1, 2, 3]
# x = x + [4] creates NEW list, y still points to old one

In [ ]:
# What will this print?
x = [1, 2, 3]
y = x
x += [4]
print(y)  # ?

In [ ]:
# Answer: [1, 2, 3, 4]
# x += [4] modifies in-place, y points to same list

---
# 💻 EXERCISES
---

In [ ]:
# Exercise 1: Implement a function that deep-copies without using copy module
def my_deepcopy(obj):
    # Handle: int, float, str, bool, None, list, dict, tuple, set
    # YOUR CODE HERE
    pass

# Tests
original = [[1, 2], {"a": [3, 4]}, (5, [6])]
copied = my_deepcopy(original)
copied[0].append(99)
assert original[0] == [1, 2]  # Not affected!
print("\u2705 Exercise 1 passed!")

In [ ]:
# Solution 1
def my_deepcopy(obj):
    if isinstance(obj, (int, float, str, bool, type(None))):
        return obj
    elif isinstance(obj, list):
        return [my_deepcopy(item) for item in obj]
    elif isinstance(obj, dict):
        return {my_deepcopy(k): my_deepcopy(v) for k, v in obj.items()}
    elif isinstance(obj, tuple):
        return tuple(my_deepcopy(item) for item in obj)
    elif isinstance(obj, set):
        return {my_deepcopy(item) for item in obj}
    return obj

original = [[1, 2], {"a": [3, 4]}, (5, [6])]
copied = my_deepcopy(original)
copied[0].append(99)
assert original[0] == [1, 2]
print("\u2705 Exercise 1 passed!")

In [ ]:
# Exercise 2: Find all falsy values in a list
def find_falsy(lst):
    # Return list of all falsy values
    # YOUR CODE HERE
    pass

# Tests
test = [0, 1, "", "hello", [], [1], {}, None, False, True, 0.0, 0j, set()]
assert find_falsy(test) == [0, "", [], {}, None, False, 0.0, 0j, set()]
print("\u2705 Exercise 2 passed!")

In [ ]:
# Solution 2
def find_falsy(lst):
    return [x for x in lst if not x]

test = [0, 1, "", "hello", [], [1], {}, None, False, True, 0.0, 0j, set()]
assert find_falsy(test) == [0, "", [], {}, None, False, 0.0, 0j, set()]
print("\u2705 Exercise 2 passed!")

---
# \u26a1 SPEED ROUND
---

In [ ]:
# \u26a1 Q1: GIL affects threads or processes?

In [ ]:
print("\u2705 Threads")

In [ ]:
# \u26a1 Q2: Shallow copy shares nested?

In [ ]:
print("\u2705 Yes!")

In [ ]:
# \u26a1 Q3: is vs == for None?

In [ ]:
print("\u2705 Use is")

In [ ]:
# \u26a1 Q4: def f(x=[]): trap?

In [ ]:
print("\u2705 Default list shared across calls")

In [ ]:
# \u26a1 Q5: Integer cache range?

In [ ]:
print("\u2705 -5 to 256")

In [ ]:
# \u26a1 Q6: a=b=[]: same list?

In [ ]:
print("\u2705 Yes!")

In [ ]:
# \u26a1 Q7: +=  on list: in-place?

In [ ]:
print("\u2705 Yes")

In [ ]:
# \u26a1 Q8: = ... + on list: in-place?

In [ ]:
print("\u2705 No (new list)")

In [ ]:
# \u26a1 Q9: range(0) truthy?

In [ ]:
print("\u2705 No (falsy)")

In [ ]:
# \u26a1 Q10: type(...) is?

In [ ]:
print("\u2705 ellipsis")

---
# 📝 QUIZ (25 Questions)
---

In [ ]:
# Q1 🟡: GIL affects?
# A) Threading  B) Multiprocessing  C) Both

In [ ]:
print("✅ A) Threading")
print("📖 GIL limits thread parallelism")

In [ ]:
# Q2 🟡: Shallow copy shares?
# A) Everything  B) Nested objects only

In [ ]:
print("✅ B) Nested objects")
print("📖 Outer container is new, inner objects shared")

In [ ]:
# Q3 🟢: is vs == for None?
# A) Use is  B) Use ==

In [ ]:
print("✅ A) Use is")
print("📖 is for identity check with singletons")

In [ ]:
# Q4 🔴: def f(x=[]): trap because?
# A) Syntax error  B) Default list shared

In [ ]:
print("✅ B) Default list shared")
print("📖 Mutable default created once")

In [ ]:
# Q5 🟡: Integer cache range?
# A) 0-100  B) -5 to 256  C) -128 to 127

In [ ]:
print("✅ B) -5 to 256")
print("📖 CPython caches these")

In [ ]:
# Q6 🟡: String interning for?
# A) All strings  B) Identifier-like strings

In [ ]:
print("✅ B)")
print("📖 Short strings without special chars")

In [ ]:
# Q7 🟡: __slots__ does?
# A) Adds methods  B) Prevents __dict__  C) Adds types

In [ ]:
print("✅ B)")
print("📖 No __dict__, saves memory")

In [ ]:
# Q8 🟡: Descriptor needs?
# A) __get__  B) __get__,__set__  C) Either

In [ ]:
print("✅ C) Either")
print("📖 Data descriptor has __set__, non-data has only __get__")

In [ ]:
# Q9 🟡: Metaclass of a class?
# A) object  B) type  C) class

In [ ]:
print("✅ B) type")
print("📖 type is the default metaclass")

In [ ]:
# Q10 🟡: *args captures as?
# A) list  B) tuple

In [ ]:
print("✅ B) tuple")
print("📖 Always tuple")

In [ ]:
# Q11 🟡: a,*b,c=[1,2,3,4,5]; type(b)?
# A) list  B) tuple

In [ ]:
print("✅ A) list")
print("📖 Extended unpacking gives list")

In [ ]:
# Q12 🟢: 1<x<10 is valid Python?
# A) Yes  B) No

In [ ]:
print("✅ A) Yes")
print("📖 Chained comparison")

In [ ]:
# Q13 🟡: Walrus operator :=?
# A) Assign  B) Assign and return

In [ ]:
print("✅ B)")
print("📖 Assignment expression")

In [ ]:
# Q14 🟡: Ellipsis type?
# A) None  B) ellipsis  C) bool

In [ ]:
print("✅ B) ellipsis")
print("📖 type(...) = ellipsis")

In [ ]:
# Q15 🟡: _ variable conventional use?
# A) Private  B) Don't care  C) Both

In [ ]:
print("✅ C) Both")
print("📖 Used for throwaway and in REPL")

In [ ]:
# Q16 🔴: a=b=[] then a.append(1); b?
# A) []  B) [1]

In [ ]:
print("✅ B) [1]")
print("📖 Same list reference")

In [ ]:
# Q17 🟡: range(0) is falsy?
# A) Yes  B) No

In [ ]:
print("✅ A) Yes")
print("📖 Empty range is falsy")

In [ ]:
# Q18 🟡: sys.getrefcount(x) usually > expected?
# A) Yes  B) No

In [ ]:
print("✅ A) Yes")
print("📖 Passing to function adds a reference")

In [ ]:
# Q19 🟡: weakref prevents GC?
# A) Yes  B) No

In [ ]:
print("✅ B) No")
print("📖 Weak references don't prevent GC")

In [ ]:
# Q20 🟡: functools.partial does?
# A) Fills some arguments  B) Removes arguments

In [ ]:
print("✅ A)")
print("📖 Pre-fills some function arguments")

In [ ]:
# Q21 🟡: operator.itemgetter(1)?
# A) Gets index 1  B) Gets item named 1

In [ ]:
print("✅ A)")
print("📖 Returns callable that gets item at index")

In [ ]:
# Q22 🟡: functools.reduce needs?
# A) 1 iterable  B) Function and iterable

In [ ]:
print("✅ B)")
print("📖 reduce(func, iterable)")

In [ ]:
# Q23 🟡: __all__ affects?
# A) All imports  B) from X import * only

In [ ]:
print("✅ B)")
print("📖 Only controls star imports")

In [ ]:
# Q24 🟡: Python GC handles?
# A) All memory  B) Circular references

In [ ]:
print("✅ B)")
print("📖 Reference counting handles most; GC handles cycles")

In [ ]:
# Q25 🔴: Late binding: [lambda:i for i in range(3)]?
# A) [0,1,2]  B) [2,2,2]

In [ ]:
print("✅ B) [2,2,2]")
print("📖 All lambdas see final value of i")

---
# 🔑 KEY TAKEAWAYS
---
1. **GIL** prevents true thread parallelism in CPython
2. **Shallow copy** shares nested objects; **deep copy** is fully independent
3. **Mutable default argument** trap — use `None` instead
4. **Integer caching**: -5 to 256 are cached singletons
5. **Late binding in closures** — fix with `lambda i=i: i`
6. **`a = b = []`** — both reference same list!
7. **`weakref`** — reference that doesn't prevent garbage collection
8. **`functools.partial`** — pre-fill function arguments